<a href="https://colab.research.google.com/github/DrKroll/llm-humanidades-salamanca/blob/main/notebooks/taller_llm/02_del_texto_al_gpt_diminuto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# 02 · Del texto al GPT diminuto: tokenizador, entrenamiento desde cero y muestreo

⏱️ **≈ 50 min** (martes 15:00–15:50)

| Parte | Contenido | Tiempo |
|---|---|---|
| A | El tokenizador: de texto a números (BPE) | ≈ 15 min |
| B | Preentrenar un GPT diminuto desde cero (2–4 min de entrenamiento en T4) | ≈ 25 min |
| C | Laboratorio de muestreo: temperatura, top-k, top-p | ≈ 10 min |

Cada participante entrena en **su propia GPU T4 de Colab**. Las secciones marcadas con ⭐ **Opcional** se pueden saltar.

In [ ]:
#@title ⚙️ Preparación: ejecuta esta celda primero (no hace falta leer el código) { display-mode: "form" }
# ⚙️ Preparación: ejecuta esta celda primero (Mayús + Intro)
import os, sys
REPO_URL = "https://github.com/DrKroll/llm-humanidades-salamanca"
REPO_DIR = "llm-humanidades-salamanca"

if "google.colab" in sys.modules:
    # En Colab: descargamos el repositorio e instalamos el paquete del taller
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL}
    os.chdir(REPO_DIR)
    !pip install -q -e .
else:
    # En tu ordenador: subimos de carpeta hasta encontrar config.yaml
    while not os.path.exists("config.yaml") and os.getcwd() != "/":
        os.chdir("..")
    if os.path.abspath("src") not in sys.path:
        sys.path.insert(0, os.path.abspath("src"))
print("Carpeta de trabajo:", os.getcwd())

## 0. ¿Tenemos GPU?
Si no aparece ✅, sigue las instrucciones y vuelve a ejecutar desde arriba.

In [ ]:
from dhllm import taller
taller.comprobar_gpu();

# Parte A · El tokenizador: de texto a números

**Objetivo:** entrenar un tokenizador **BPE** (*Byte Pair Encoding*) con tus textos y ver cómo trocea el lenguaje.

Un ordenador no entiende letras, solo números. El **tokenizador** divide el texto en trozos
(**tokens**) y asigna un número a cada uno. Opciones:

| Unidad | Ejemplo con «caballero» | Problema |
|---|---|---|
| letras | c·a·b·a·l·l·e·r·o | secuencias muy largas |
| palabras | caballero | vocabulario enorme, palabras nuevas desconocidas |
| **subpalabras (BPE)** | caball·ero | ✅ el compromiso que usan GPT, Llama, etc. |

**Cómo funciona BPE:** empieza con caracteres sueltos y une una y otra vez el par más frecuente
(«q»+«u» → «qu», «qu»+«e» → «que»...) hasta alcanzar el tamaño de vocabulario deseado.

## ⚙️ Ajustes
Los textos se toman de `config.yaml` (por defecto, los de ejemplo; ver cuaderno 01).

In [ ]:
VOCABULARIO = 4000   # prueba 500, 2000, 8000
FRASE = "En un lugar de la Mancha, de cuyo nombre no quiero acordarme, no ha mucho tiempo que vivía un hidalgo."

## 1. Entrenar el tokenizador y ver los tokens
Tarda unos segundos. Cada color es un token; el espacio suele ir pegado al principio de la palabra.

In [ ]:
tok = taller.entrenar_tokenizador(VOCABULARIO, FRASE)

Y estos son los números que «ve» el modelo:

In [ ]:
ids = tok.encode(FRASE)
print(ids)
print("De vuelta a texto:", tok.decode(ids))

## 2. ¿Qué ha aprendido el tokenizador?
Los tokens más largos del vocabulario suelen ser palabras frecuentes de tu corpus.

In [ ]:
from dhllm.tokenizer import show_vocabulary_sample, show_tokens
show_vocabulary_sample(tok, n=40);

> ⭐ **Opcional** — puedes saltarte esta sección si vas justo de tiempo.

## 3. Comparar con tokenizadores «de verdad»
GPT-2 se entrenó sobre todo con inglés; Qwen2.5, con muchos idiomas.
**Menos tokens por palabra = más eficiente** para ese idioma. (Necesita conexión a internet.)

In [ ]:
taller.comparar_tokenizadores(tok, FRASE)

## 🖐️ Tu turno
1. Prueba `VOCABULARIO = 300` y `VOCABULARIO = 10000`. ¿Qué cambia en los colores de la frase?
2. Tokeniza una frase en otro idioma o con ortografía antigua: `show_tokens(tok, "Quiso la fortuna...")`.
3. **Para pensar:** si un idioma necesita el doble de tokens que el inglés, ¿qué consecuencias tiene
   en el coste y la calidad de los grandes modelos comerciales?

# Parte B · Preentrenar un GPT diminuto desde cero

**Objetivo:** entrenar tu propio modelo de lenguaje (¡con pesos aleatorios al principio!) usando solo tus textos.

La tarea es siempre la misma: **predecir el siguiente token**. El modelo ve un trozo de texto y
tiene que adivinar qué viene después. Al principio adivina al azar; poco a poco mejora.

- La **pérdida** (*loss*) mide cuánto se equivoca. Debe **bajar** durante el entrenamiento.
- **Entrenamiento** vs. **validación**: la validación es texto que el modelo no ve al entrenar.
  Si la pérdida de entrenamiento baja pero la de validación sube, el modelo está **memorizando** (*sobreajuste*).

Nuestro modelo tiene ~2,6 millones de parámetros. GPT-3 tiene 175 000 millones. 🙂
Inspirado en [nanoGPT](https://github.com/karpathy/nanoGPT) y [minimind](https://github.com/jingyaogong/minimind).

⏱️ En tu GPU T4 de Colab: unos 2–4 minutos. (Sin GPU se reduce solo a 300 pasos.)

## ⚙️ Ajustes
Solo tocamos lo esencial. El resto está en `config.yaml` (sección `tiny_gpt`).

In [ ]:
AJUSTES = {"pasos": 1500, "capas": 4}   # más capas = modelo más grande (y más lento)
INICIO = "En un lugar de la Mancha"     # texto con el que el modelo empezará a escribir

## 1. Preparar: textos → números, y un modelo «vacío»
Se usa el tokenizador de la parte A.

In [ ]:
gpt = taller.preparar_gpt(AJUSTES)

Antes de entrenar, el modelo escribe sin sentido (sus pesos son aleatorios):

In [ ]:
taller.escribir_gpt(gpt, INICIO)

## 2. ¡Entrenar!
Verás una **barra de progreso**, la pérdida cada cierto número de pasos y, de vez en cuando, una muestra de texto.
Al final: la curva de pérdida, cómo ha mejorado el texto y el modelo guardado en `outputs/tiny_gpt`.

**Interpretación de la pérdida:** ~8,3 significa «adivinar al azar» entre 4000 tokens (ln 4000 ≈ 8,3).
Si baja a ~4, el modelo duda en promedio entre unas e⁴ ≈ 55 opciones.

In [ ]:
historial = taller.entrenar_gpt(gpt, INICIO)

## 3. Generar texto nuevo

In [ ]:
taller.escribir_gpt(gpt, INICIO, n=3, tokens=80)

> 🔍 **¿Quieres mirar el código?** El modelo está en [`src/dhllm/model.py`](../../src/dhllm/model.py) (~100 líneas,
> estilo nanoGPT) y el bucle de entrenamiento en [`src/dhllm/pretrain.py`](../../src/dhllm/pretrain.py).
> En Colab: icono de carpeta 📁 → `llm-humanidades-salamanca/src/dhllm/`.

## 🖐️ Tu turno
1. Cambia `INICIO` por otra frase («Sancho dijo», «Yo por bien tengo»...).

⭐ **Opcional (en casa o si vas rápido; cada prueba vuelve a entrenar):**

2. Entrena con más pasos (`"pasos": 3000`). ¿Mejora la validación o empieza a subir?
3. Haz el modelo más grande (`"capas": 6, "dimension": 256`) o más pequeño (`"capas": 2`). Compara curvas.
4. **Para pensar:** ¿«entiende» este modelo el Quijote? ¿Qué ha aprendido exactamente?

# Parte C · Laboratorio de muestreo: temperatura, top-k y top-p

**Objetivo:** entender cómo un modelo **elige** la siguiente palabra.

En cada paso, el modelo calcula una probabilidad para **cada** token del vocabulario. Después hay que
elegir uno:

- **Greedy** (temperatura 0): siempre el más probable → repetitivo.
- **Temperatura**: < 1 hace la distribución más «puntiaguda» (conservador); > 1 la aplana (creativo, caótico).
- **Top-k**: solo se sortea entre los *k* tokens más probables.
- **Top-p** (*nucleus sampling*): solo entre los tokens más probables que suman una probabilidad *p*.

Usamos el GPT diminuto de la parte B (si no existe, se carga SmolLM2-135M).

⏱️ ≈ 10 min. Si vas justo de tiempo, haz solo los puntos 1 y 3.

## ⚙️ Ajustes

In [ ]:
from dhllm.sampling import next_token_table, compare_settings, generate
INICIO = "Dijo don Quijote"
modelo, tok = taller.cargar_para_muestreo("auto")   # o un nombre de Hugging Face

## 1. ¿Qué token viene después?
Estas son las probabilidades del modelo para el siguiente token.

In [ ]:
next_token_table(modelo, tok, INICIO, temperature=1.0, n=15);

## 2. El efecto de la temperatura
Misma frase, distinta temperatura. Fíjate en cómo cambian las barras.

In [ ]:
for t in [0.3, 1.0, 2.0]:
    next_token_table(modelo, tok, INICIO, temperature=t, n=10)

## 3. Generar con distintos ajustes
greedy · equilibrado · caótico · solo 5 opciones

In [ ]:
compare_settings(modelo, tok, INICIO, [
    {"temperature": 0}, {"temperature": 0.7, "top_k": 50, "top_p": 0.95},
    {"temperature": 1.5, "top_k": None, "top_p": 1.0}, {"temperature": 1.0, "top_k": 5, "top_p": 1.0},
], max_new_tokens=60);

## 4. Tu propio experimento
Cambia los tres números (`top_k=None` o `top_p=1.0` = sin límite).

In [ ]:
print(generate(modelo, tok, INICIO, max_new_tokens=120, temperature=0.9, top_k=40, top_p=0.9, seed=1))

## 🖐️ Tu turno
1. Encuentra los ajustes que producen el texto más «cervantino». ¿Y el más absurdo?
2. Con `temperature=0`, ¿se repite el texto? ¿Por qué?
3. Ejecuta dos veces con `seed=None`. ¿Por qué cambia el resultado?
4. **Para pensar:** un chatbot responde distinto a la misma pregunta. ¿Qué implica esto para citar
   o usar textos generados en investigación?